In [1]:
from correlation_stability import run_stability_analysis
import scanpy as sc
from run_dgcna_correlation import run_dgcna_analysis
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:

adata=sc.read_loom('Integrated_betas.loom') 
adata.obs_names=adata.obs.obs_names
adata.var_names=adata.var.gene_name



C:\Users\Leo\anaconda3\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
C:\Users\Leo\anaconda3\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [3]:
#adata=adata[adata.obs.Disease=='T2D'].copy()

import numpy as np

np.random.seed(0)  # optional, for reproducibility
n = adata.n_obs
groups = np.array(['A'] * (n // 2) + ['B'] * (n - n // 2))
np.random.shuffle(groups)

adata.obs['Group'] = groups

In [4]:
sel_genes=pd.read_excel("sel_genes.xlsx")
ndcg_genes=sel_genes.Gene_name.unique()
glyco=sel_genes[sel_genes.Module=='(6) Glycolysis'].Gene_name.values

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
import random

# Keep ALL subjects only

# Subject/donor ID
subject_col = 'Donor'  # <-- change if needed

subjects = np.array(adata.obs[subject_col].unique())

assert len(subjects) == 17, f"Expected 17 subjects, found {len(subjects)}"

summary = []

# 9-vs-8 splits
# Keep only one of each A/B vs B/A pair
splits = list(combinations(subjects, 9))

# Pick only 100 splits
n_random_splits = 30
random_seed = 42
random.seed(random_seed)
splits = random.sample(splits, min(n_random_splits, len(splits)))


print(f"Running {len(splits)} unique subject splits...")

for it, group_A_subjects in enumerate(splits):

    group_A_subjects = set(group_A_subjects)

    # Assign all cells from each subject to the same group
    adata.obs['Group'] = np.where(
        adata.obs[subject_col].isin(group_A_subjects),
        'A',
        'B'
    )

    # Run complete dGCNA pipeline
    results = run_dgcna_analysis(
        adata,
        genes=ndcg_genes,
        disease_col='Group',
        t2d_value='B',
        ctrl_value='A',
        n_bootstrap=512,
        threshold=0.975
    )

    corr = results['filtered_net']

    # Upper triangle only
    upper = np.triu(corr.values, k=1)
    sig_delta = np.abs(upper[upper != 0])

    # Total number of tested gene pairs
    n_genes = corr.shape[0]
    n_total = n_genes * (n_genes - 1) // 2

    summary.append({
        'iteration': it + 1,
        'group_A': ','.join(sorted(group_A_subjects)),
        'group_B': ','.join(sorted(set(subjects) - group_A_subjects)),
        'n_sig': len(sig_delta),
        'pct_sig': 100 * len(sig_delta) / n_total,
        'median_abs_delta_r': (
            np.median(sig_delta) if len(sig_delta) else np.nan
        )
    })

summary_df = pd.DataFrame(summary)

# Save results
summary_df.to_csv(
    "dGCNA_random_9vs8_splits.csv",
    index=False
)

print(f"Completed {len(summary_df)} unique 9-vs-8 splits.")

Running 30 unique subject splits...
Using 664 / 671 specified genes

=== Step 1: Computing LMM residuals (per condition) ===
Cells: 541, donors: 8, genes: 664
Cells: 497, donors: 9, genes: 664

=== Step 2: Building differential network ===
T2D cells: 541, Control cells: 497

=== Step 3: Bootstrap filtering ===


Random donor groups (with replacement): 100%|████████████████████████████████████████| 512/512 [01:16<00:00,  6.73it/s]


Significant edges: 11559 / 220116 (5.3%)
Using 664 / 671 specified genes

=== Step 1: Computing LMM residuals (per condition) ===
Cells: 407, donors: 8, genes: 664
Cells: 631, donors: 9, genes: 664

=== Step 2: Building differential network ===
T2D cells: 407, Control cells: 631

=== Step 3: Bootstrap filtering ===


Random donor groups (with replacement): 100%|████████████████████████████████████████| 512/512 [01:19<00:00,  6.46it/s]


Significant edges: 11089 / 220116 (5.0%)
Using 664 / 671 specified genes

=== Step 1: Computing LMM residuals (per condition) ===
Cells: 499, donors: 8, genes: 664
Cells: 539, donors: 9, genes: 664

=== Step 2: Building differential network ===
T2D cells: 499, Control cells: 539

=== Step 3: Bootstrap filtering ===


...

Random donor groups (with replacement): 100%|████████████████████████████████████████| 512/512 [01:12<00:00,  7.08it/s]


Significant edges: 14433 / 220116 (6.6%)
Using 664 / 671 specified genes

=== Step 1: Computing LMM residuals (per condition) ===
Cells: 493, donors: 8, genes: 664
Cells: 545, donors: 9, genes: 664

=== Step 2: Building differential network ===
T2D cells: 493, Control cells: 545

=== Step 3: Bootstrap filtering ===


Random donor groups (with replacement): 100%|████████████████████████████████████████| 512/512 [01:13<00:00,  7.01it/s]


Significant edges: 11409 / 220116 (5.2%)
Using 664 / 671 specified genes

=== Step 1: Computing LMM residuals (per condition) ===
Cells: 488, donors: 8, genes: 664
Cells: 550, donors: 9, genes: 664

=== Step 2: Building differential network ===
T2D cells: 488, Control cells: 550

=== Step 3: Bootstrap filtering ===


Random donor groups (with replacement):  73%|█████████████████████████████▍          | 376/512 [00:57<00:20,  6.70it/s]

In [ ]:

# Summary
print("\nSummary:")
print(f"Number of unique splits: {len(summary_df)}")
print(
    f"Significant edges — median: "
    f"{summary_df['n_sig'].median():.0f}"
)
print(
    f"Significant edges — range: "
    f"{summary_df['n_sig'].min():.0f}–"
    f"{summary_df['n_sig'].max():.0f}"
)
print(
    f"Percentage significant — median: "
    f"{summary_df['pct_sig'].median():.1f}%"
)
print(
    f"Percentage significant — range: "
    f"{summary_df['pct_sig'].min():.1f}–"
    f"{summary_df['pct_sig'].max():.1f}%"
)
print(
    f"Median |Δr| across splits: "
    f"{summary_df['median_abs_delta_r'].median():.3f}"
)


In [ ]:
# Distribution of % of significant edges on random shuffles of control donors
fig, ax = plt.subplots(figsize=(5, 5))

sns.boxplot(summary_df, y='pct_sig')
sns.stripplot(summary_df, y='pct_sig',  linewidth=1, color='w', s=10, edgecolor='k')

ax.set_xlabel(f'Random shuffles of Control subjects')
ax.set_ylabel(f'Percentage of significant edges')
#ax.set_title(f'CTRL Sample ({largest_ctrl_donor}, n={largest_ctrl_n})\nr = {r_resid_ctrl:.3f}')
ax.spines[['top', 'right']].set_visible(False)
ax.legend()
sns.despine()
plt.tight_layout()
#plt.savefig('Figure3/pct_significant_T2D.png')
#plt.savefig('Figure3/pct_significant_T2D.svg')

In [ ]:
summary_df